# Loan Approval Prediction System

## MLOps End Term Project

This notebook covers data ingestion, exploration, validation,
feature engineering, model training and evaluation.

In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully! ✅")

Libraries imported successfully! ✅


In [2]:
df = pd.read_csv("../data/raw/loan_data.csv")

print("Dataset loaded successfully! ✅")
print("Shape:", df.shape)

Dataset loaded successfully! ✅


Shape: (614, 13)


In [3]:
df.head()


,loan_id,gender,married,dependents,education,self_employed,applicantincome,coapplicantincome,loanamount,loan_amount_term,credit_history,property_area,loan_status
0,lp001002,male,no,0,graduate,no,5849,0.0,NaN,360.0,1.0,urban,y
1,lp001003,male,yes,1,graduate,no,4583,1508.0,128.0,360.0,1.0,rural,n
2,lp001005,male,yes,0,graduate,yes,3000,0.0,66.0,360.0,1.0,urban,y
3,lp001006,male,yes,0,not graduate,no,2583,2358.0,120.0,360.0,1.0,urban,y
4,lp001008,male,no,0,graduate,no,6000,0.0,141.0,360.0,1.0,urban,y


In [4]:
df.columns.tolist()

['loan_id',
 'gender',
 'married',
 'dependents',
 'education',
 'self_employed',
 'applicantincome',
 'coapplicantincome',
 'loanamount',
 'loan_amount_term',
 'credit_history',
 'property_area',
 'loan_status']

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   loan_id            614 non-null    str    
 1   gender             601 non-null    str    
 2   married            611 non-null    str    
 3   dependents         599 non-null    str    
 4   education          614 non-null    str    
 5   self_employed      582 non-null    str    
 6   applicantincome    614 non-null    int64  
 7   coapplicantincome  614 non-null    float64
 8   loanamount         592 non-null    float64
 9   loan_amount_term   600 non-null    float64
 10  credit_history     564 non-null    float64
 11  property_area      614 non-null    str    
 12  loan_status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 83.4 KB


In [6]:
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)


Missing values in each column:
loan_id               0
gender               13
married               3
dependents           15
education             0
self_employed        32
applicantincome       0
coapplicantincome     0
loanamount           22
loan_amount_term     14
credit_history       50
property_area         0
loan_status           0
dtype: int64


In [7]:
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 0


In [8]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
loan_id,614,614,lp001002,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,601,2,male,489,NaN,NaN,NaN,NaN,NaN,NaN,NaN
married,611,2,yes,398,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dependents,599,4,0,345,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,614,2,graduate,480,NaN,NaN,NaN,NaN,NaN,NaN,NaN
self_employed,582,2,no,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
applicantincome,614.0,NaN,NaN,NaN,5403.459283,6109.041673,150.0,2877.5,3812.5,5795.0,81000.0
coapplicantincome,614.0,NaN,NaN,NaN,1621.245798,2926.248369,0.0,0.0,1188.5,2297.25,41667.0
loanamount,592.0,NaN,NaN,NaN,146.412162,85.587325,9.0,100.0,128.0,168.0,700.0
loan_amount_term,600.0,NaN,NaN,NaN,342.0,65.12041,12.0,360.0,360.0,360.0,480.0


In [9]:
print("Loan Status Distribution:")
print(df["loan_status"].value_counts())

print("\nLoan Status Percentage:")
print(df["loan_status"].value_counts(normalize=True) * 100)

Loan Status Distribution:
loan_status
y    422
n    192
Name: count, dtype: int64

Loan Status Percentage:
loan_status
y    68.729642
n    31.270358
Name: proportion, dtype: float64


In [10]:
categorical_columns = [
    "gender",
    "married",
    "dependents",
    "education",
    "self_employed",
    "property_area"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False))


--- gender ---
gender
male      489
female    112
NaN        13
Name: count, dtype: int64

--- married ---
married
yes    398
no     213
NaN      3
Name: count, dtype: int64

--- dependents ---
dependents
0      345
1      102
2      101
3+      51
NaN     15
Name: count, dtype: int64

--- education ---
education
graduate        480
not graduate    134
Name: count, dtype: int64

--- self_employed ---
self_employed
no     500
yes     82
NaN     32
Name: count, dtype: int64

--- property_area ---
property_area
semiurban    233
urban        202
rural        179
Name: count, dtype: int64


In [11]:
numerical_columns = [
    "applicantincome",
    "coapplicantincome",
    "loanamount",
    "loan_amount_term",
    "credit_history"
]

df[numerical_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
applicantincome,614.0,5403.459283,6109.041673,150.0,2877.5,3812.5,5795.00,81000.0
coapplicantincome,614.0,1621.245798,2926.248369,0.0,0.0,1188.5,2297.25,41667.0
loanamount,592.0,146.412162,85.587325,9.0,100.0,128.0,168.00,700.0
loan_amount_term,600.0,342.000000,65.120410,12.0,360.0,360.0,360.00,480.0
credit_history,564.0,0.842199,0.364878,0.0,1.0,1.0,1.00,1.0


In [12]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())
print("Total missing values:", df.isnull().sum().sum())

Number of rows: 614
Number of columns: 13


Duplicate rows: 0
Total missing values: 149


In [13]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: C:\Users\ashbi\OneDrive\Desktop\loan-approval-mlops


In [14]:
from src.validation.data_validation import validate_dataset

validate_dataset(df)

Data validation passed successfully! ✅


True

## Data Transformation & Feature Engineering

The dataset contains numerical and categorical features.
Missing values will be handled using appropriate imputation,
categorical features will be encoded, and the target variable
will be converted into a binary format for model training.

In [15]:
from src.transformation.data_transformation import (
    split_features_and_target,
    train_test_split_data,
    save_processed_datasets,
    build_preprocessing_pipeline,
)

# Separate raw dataframe into model features (X) and target (y).
# loan_id is dropped inside split_features_and_target since it is only
# an identifier, and loan_status is mapped from y/n to 1/0.
X, y = split_features_and_target(df)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())


Features shape: (614, 11)
Target shape: (614,)

Target distribution:
loan_status
1    422
0    192
Name: count, dtype: int64


## Train/Test Split

The split happens **before** any imputation/encoding is fit, so no
information from the test set leaks into the preprocessing statistics
(e.g. median/most-frequent values). The split is stratified on
`loan_status` to preserve the ~69%/31% class balance in both sets.


In [16]:
X_train, X_test, y_train, y_test = train_test_split_data(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

# Persist the split (pre-encoding) for reproducibility / reuse outside the notebook.
save_processed_datasets(X_train, X_test, y_train, y_test, "../data/processed")
print("\nProcessed train/test CSVs saved to data/processed/ ✅")


X_train shape: (491, 11)
X_test shape: (123, 11)
y_train shape: (491,)
y_test shape: (123,)

Train target distribution:
loan_status
1    0.686354
0    0.313646
Name: proportion, dtype: float64

Test target distribution:
loan_status
1    0.691057
0    0.308943
Name: proportion, dtype: float64

Processed train/test CSVs saved to data/processed/ ✅


## Preprocessing Pipeline

The `ColumnTransformer` below imputes missing values (median for
numerical columns, most-frequent for categorical columns) and
one-hot encodes categorical columns. It is combined with each model
inside a single `Pipeline` during training, so it is fit **only** on
`X_train` and applied unchanged to `X_test` (and later, to new API
inputs) — this avoids data leakage.


In [17]:
preprocessor = build_preprocessing_pipeline()
preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

## Model Training

Two candidate models are trained on the identical preprocessed
training data so they can be compared fairly:

1. Logistic Regression
2. Random Forest Classifier

Each model is wrapped together with the preprocessing pipeline in a
single `sklearn.Pipeline`, and `random_state=42` is used wherever
applicable for reproducibility.


In [18]:
from src.training.model_training import (
    train_models,
    evaluate_models,
    build_comparison_table,
    select_best_model,
    save_model_artifact,
    load_model_artifact,
)

fitted_models = train_models(X_train, y_train)

for model_name in fitted_models:
    print(f"{model_name} trained successfully! ✅")


Logistic Regression trained successfully! ✅
Random Forest trained successfully! ✅


## Model Evaluation

Each model is evaluated on the held-out test set using accuracy,
precision, recall, F1-score, and a confusion matrix. Accuracy alone
is not used to pick the best model since `loan_status` is imbalanced
(~69% approved / 31% rejected).


In [19]:
results = evaluate_models(fitted_models, X_test, y_test)

comparison_table = build_comparison_table(results)
print("Model Comparison:")
comparison_table


Model Comparison:


,accuracy,precision,recall,f1_score
model,,,,
Logistic Regression,0.8618,0.8400,0.9882,0.9081
Random Forest,0.8211,0.8462,0.9059,0.8750


In [20]:
for model_name, metrics in results.items():
    print(f"\n--- {model_name}: Confusion Matrix ---")
    print(metrics["confusion_matrix"])



--- Logistic Regression: Confusion Matrix ---
[[22 16]
 [ 1 84]]

--- Random Forest: Confusion Matrix ---
[[24 14]
 [ 8 77]]


## Final Model Saving

The best model is selected by F1-score (a balance of precision and
recall). The saved artifact is a single `Pipeline` containing both
the preprocessing steps and the trained classifier, so the exact same
preprocessing is applied automatically to any new input later (e.g.
from the API).


In [21]:
best_model_name, best_pipeline = select_best_model(results, fitted_models, metric="f1_score")

print(f"Best model: {best_model_name}")
print(f"F1-score: {results[best_model_name]['f1_score']:.4f}")

model_output_path = "../models/loan_approval_model.joblib"
save_model_artifact(best_pipeline, model_output_path)
print(f"\nModel saved to {model_output_path} ✅")


Best model: Logistic Regression
F1-score: 0.9081

Model saved to ../models/loan_approval_model.joblib ✅


## Verification

Load the saved artifact back from disk and confirm it can produce a
prediction on a real test-set sample.


In [22]:
loaded_pipeline = load_model_artifact(model_output_path)

sample = X_test.iloc[[0]]
prediction = loaded_pipeline.predict(sample)

print("Sample input:")
print(sample)
print("\nActual loan_status:", y_test.iloc[0])
print("Predicted loan_status:", prediction[0])
print("\nModel loaded and test prediction successful! ✅")


Sample input:
    gender married dependents education self_employed  applicantincome  \
150   male      no          0  graduate            no             6277   

     coapplicantincome  loanamount  loan_amount_term  credit_history  \
150                0.0       118.0             360.0             0.0   

    property_area  
150         rural  

Actual loan_status: 0
Predicted loan_status: 0

Model loaded and test prediction successful! ✅
